# Optimized Whisper on AWS Neuron (Trainium2 / Inferentia2)

This notebook runs OpenAI's [Whisper large-v3-turbo](https://huggingface.co/openai/whisper-large-v3-turbo) on AWS Neuron instances using [NxD Inference](https://github.com/aws-neuron/neuronx-distributed-inference) with four optimizations:

1. **Cross-attention K/V cache** — eliminates redundant K/V projections (19.7B FLOPs/token) and audio transfer (3.84MB/token) during decode
2. **Fused QKV projections** — replaces 3 separate Q/K/V linear layers with a single fused matmul in self-attention
3. **NKI flash attention** — replaces matmul-based attention with hardware-accelerated flash attention in all 32 encoder layers
4. **NKI fused Conv1D+GELU** — replaces separate Conv1D and GELU ops with a single fused NKI kernel in the encoder frontend

**Supported instances:** trn2.3xlarge, inf2.xlarge (or any Neuron instance with ≥1 NeuronCore)

**Model:** whisper-large-v3-turbo (809M params, 32 encoder layers, 4 decoder layers) at TP=1, FP16

## 1. Setup

Activate the pre-installed PyTorch Neuron environment and install dependencies.

```bash
source /opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/bin/activate
```

In [1]:
!pip install -q openai-whisper soundfile librosa gtts jiwer 2>/dev/null
!sudo add-apt-repository -y universe > /dev/null 2>&1 && sudo apt-get update -qq > /dev/null 2>&1 && sudo apt-get install -y -qq ffmpeg > /dev/null 2>&1
!which ffmpeg

/usr/bin/ffmpeg


## 2. Install Optimized NxDI

Clone our optimized fork of neuronx-distributed-inference and install it over the system package.
This replaces only the Whisper model files — all other NxDI models are unaffected.

**Note:** If running via `run_notebook.sh`, these packages are pre-installed. The cells below
detect this and skip the install if already present.

In [2]:
import os
import subprocess
import sys

NXDI_BRANCH = "fix/whisper-all-optimizations"
NXDI_REPO = "https://github.com/jimburtoft/neuronx-distributed-inference.git"
NXDI_CLONE_DIR = "/tmp/nxdi-optimized"

# Clone the optimized branch
if not os.path.exists(NXDI_CLONE_DIR):
    subprocess.run(
        ["git", "clone", "--branch", NXDI_BRANCH, "--single-branch", "--depth", "1", NXDI_REPO, NXDI_CLONE_DIR],
        check=True,
    )
    print(f"Cloned {NXDI_BRANCH} to {NXDI_CLONE_DIR}")
else:
    print(f"Already cloned at {NXDI_CLONE_DIR}")

# Install over the system NxDI package (if not already installed from run_notebook.sh)
try:
    import neuronx_distributed_inference
    print(f"NxDI already available: {neuronx_distributed_inference.__file__}")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", NXDI_CLONE_DIR], check=True)
    print("Optimized NxDI installed — please restart kernel if running interactively")

Already cloned at /tmp/nxdi-optimized
NxDI already available: /opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed_inference/__init__.py


## 3. Install NKI Conv1D kernel (optional)

The DLAMI includes `nkilib` but may lack the newer `conv1d` kernel. We copy it
from the nki-library repo. If unavailable, the notebook falls back to PyTorch Conv1D.

In [3]:
import os, subprocess, sys

NKI_LIB_DIR = "/tmp/nki-library"

try:
    from nkilib.experimental.conv.conv1d import conv1d as nki_conv1d
    print("NKI Conv1D kernel already available")
except ImportError:
    # Copy conv1d.py from nki-library repo into the installed nkilib package
    try:
        import nkilib.experimental.conv
        conv_dir = os.path.dirname(nkilib.experimental.conv.__file__)
        if not os.path.exists(NKI_LIB_DIR):
            subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/aws-neuron/nki-library.git", NKI_LIB_DIR],
                check=True, capture_output=True)
        src = os.path.join(NKI_LIB_DIR, "src", "nkilib_src", "nkilib", "experimental", "conv", "conv1d.py")
        dst = os.path.join(conv_dir, "conv1d.py")
        import shutil
        shutil.copy2(src, dst)
        from nkilib.experimental.conv.conv1d import conv1d as nki_conv1d
        print(f"NKI Conv1D kernel installed to {dst}")
    except Exception as e:
        print(f"NKI Conv1D not available ({e}) — will use PyTorch fallback")

NKI Conv1D not available (No module named 'nki') — will use PyTorch fallback


## 4. Detect Hardware

Auto-detect whether we're on Trainium or Inferentia and configure accordingly.

In [4]:
import time
import json
import torch

os.environ["NEURON_RT_NUM_CORES"] = "1"

# NKI conv1d kernel needs platform hint during compilation
result = subprocess.run(["neuron-ls"], capture_output=True, text=True)
neuron_ls_output = result.stdout
print(neuron_ls_output)

if "trn2" in neuron_ls_output.lower():
    PLATFORM = "trn2"
    os.environ["NEURON_PLATFORM_TARGET_OVERRIDE"] = "trn2"
elif "trn1" in neuron_ls_output.lower():
    PLATFORM = "trn1"
elif "inf2" in neuron_ls_output.lower() or "inf1" in neuron_ls_output.lower():
    PLATFORM = "inf2"
else:
    # Fallback: check instance type from metadata
    try:
        token_result = subprocess.run(
            ["curl", "-s", "-X", "PUT", "http://169.254.169.254/latest/api/token",
             "-H", "X-aws-ec2-metadata-token-ttl-seconds: 60"],
            capture_output=True, text=True, timeout=2
        )
        token = token_result.stdout.strip()
        itype_result = subprocess.run(
            ["curl", "-s", "-H", f"X-aws-ec2-metadata-token: {token}",
             "http://169.254.169.254/latest/meta-data/instance-type"],
            capture_output=True, text=True, timeout=2
        )
        instance_type = itype_result.stdout.strip()
        if "trn2" in instance_type:
            PLATFORM = "trn2"
            os.environ["NEURON_PLATFORM_TARGET_OVERRIDE"] = "trn2"
        elif "trn1" in instance_type:
            PLATFORM = "trn1"
        elif "inf" in instance_type:
            PLATFORM = "inf2"
        else:
            PLATFORM = "unknown"
    except Exception:
        PLATFORM = "unknown"

print(f"\nDetected platform: {PLATFORM}")

instance-type: trn2.3xlarge
instance-id: i-0e236283432e0a01a
logical-neuroncore-config: 2
+--------+--------+----------+--------+--------------+----------+------+
| NEURON | NEURON |  NEURON  | NEURON |     PCI      |   CPU    | NUMA |
| DEVICE | CORES  | CORE IDS | MEMORY |     BDF      | AFFINITY | NODE |
+--------+--------+----------+--------+--------------+----------+------+
| 0      | 4      | 0-3      | 96 GB  | 0000:33:00.0 | 0-11     | 0    |
+--------+--------+----------+--------+--------------+----------+------+


Detected platform: trn2


In [5]:
from neuronx_distributed_inference.models.config import NeuronConfig
from neuronx_distributed_inference.models.whisper.modeling_whisper import (
    WhisperInferenceConfig,
    NeuronApplicationWhisper,
)
from neuronx_distributed_inference.utils.hf_adapter import load_pretrained_config

/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/parallel_layers/layers.py:16: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from .mappings import (
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/parallel_layers/layers.py:16: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from .mappings import (
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/parallel_layers/layers.py:16: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from .mappings import (


/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:74: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  component, error = import_nki(config)
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:74: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  component, error = import_nki(config)
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:74: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  component, error = import_nki(config)
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:74: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  component, error = import_nki(config)
/opt/aws_neuronx

/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/modules/moe/moe_fused_tkg.py:49: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  component, error = import_nki(config)
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/modules/moe/moe_fused_tkg.py:49: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  component, error = import_nki(config)


/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed_inference/modules/attention/utils.py:13: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from neuronx_distributed_inference.modules.custom_calls import neuron_cumsum
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed_inference/modules/lora_serving/lora_checkpoint.py:9: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from neuronx_distributed_inference.modules.attention.gqa import replicate_kv
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed_inference/modules/lora_serving/lora_checkpoint.py:9: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from neuronx_distributed_inference.modules.attention.gqa import replicate_kv
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuron

## 5. Download Model

In [6]:
MODEL_PATH = "/tmp/whisper-large-v3-turbo/"
COMPILED_MODEL_PATH = f"/tmp/whisper-large-v3-turbo-neuron-{PLATFORM}/"

if not os.path.exists(os.path.join(MODEL_PATH, "config.json")):
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id="openai/whisper-large-v3-turbo",
        local_dir=MODEL_PATH,
        ignore_patterns=["*.onnx", "*.tflite", "flax_model*", "tf_model*"],
    )
    print(f"Downloaded to {MODEL_PATH}")
else:
    print(f"Model already exists at {MODEL_PATH}")

Model already exists at /tmp/whisper-large-v3-turbo/


## 6. Compile for Neuron

NxDI compiles three separate graphs:
- **Encoder** — processes the 30-second mel spectrogram window
- **Decoder Prefill** — processes all prompt tokens at once, caches encoder K/V
- **Decoder Decode** — generates one token at a time using KV cache

Compilation is a one-time cost. The compiled model is platform-specific (trn2 vs inf2).

In [7]:
neuron_config = NeuronConfig(
    batch_size=1,
    torch_dtype=torch.float16,
    tp_degree=1,
)
inference_config = WhisperInferenceConfig(
    neuron_config,
    load_config=load_pretrained_config(MODEL_PATH),
)

if not os.path.exists(COMPILED_MODEL_PATH):
    print(f"Compiling for {PLATFORM} (one-time cost, ~60-120s)...")
    t0 = time.time()
    model = NeuronApplicationWhisper(MODEL_PATH, config=inference_config)
    model.compile(COMPILED_MODEL_PATH)
    compile_time = time.time() - t0
    print(f"Compilation complete in {compile_time:.1f}s")
else:
    compile_time = None
    print(f"Using cached compiled model at {COMPILED_MODEL_PATH}")

Compiling for trn2 (one-time cost, ~60-120s)...


INFO:Neuron:Saving the neuron_config to /tmp/whisper-large-v3-turbo-neuron-trn2/encoder/


INFO:Neuron:Generating HLOs for the following models: ['Encoder']


[2026-03-11 19:56:55.823: I neuronx_distributed/parallel_layers/parallel_state.py:638] > initializing tensor model parallel with size 1


[2026-03-11 19:56:55.824: I neuronx_distributed/parallel_layers/parallel_state.py:639] > initializing pipeline model parallel with size 1


...

INFO:Neuron:Finished Compilation for all HLOs in 15.495153427124023 seconds


INFO:Neuron:Can't find a priority model, falling back to the existing weight layout


INFO:Neuron:Finished building model in 17.07803463935852 seconds


INFO:Neuron:SKIPPING pre-sharding the checkpoints. The checkpoints will be sharded during load time.


Compilation complete in 27.9s


## 7. Load Compiled Model

In [8]:
t0 = time.time()
model = NeuronApplicationWhisper(COMPILED_MODEL_PATH, config=inference_config)
model.load(COMPILED_MODEL_PATH)
load_time = time.time() - t0
print(f"Model loaded in {load_time:.1f}s")

INFO:Neuron:Sharding weights on load...


INFO:Neuron:Sharding weights for ranks: 0...0


[2026-03-11 19:57:23.741: I neuronx_distributed/parallel_layers/parallel_state.py:638] > initializing tensor model parallel with size 1


[2026-03-11 19:57:23.741: I neuronx_distributed/parallel_layers/parallel_state.py:639] > initializing pipeline model parallel with size 1


[2026-03-11 19:57:23.742: I neuronx_distributed/parallel_layers/parallel_state.py:640] > initializing context model parallel with size 1


...

INFO:Neuron:Done Sharding weights in 0.06537808699977177


INFO:Neuron:Finished weights loading in 0.31598757000028854 seconds


INFO:Neuron:Warming up the model.


INFO:Neuron:Warmup completed in 0.09158134460449219 seconds.


Model loaded in 7.7s


## 8. Generate Test Audio

Create three speech samples of different lengths using Google Text-to-Speech.

In [9]:
import shutil
import soundfile as sf
import numpy as np
from gtts import gTTS

FFMPEG = shutil.which("ffmpeg") or "/usr/bin/ffmpeg"
AUDIO_DIR = "/tmp/whisper_audio"
os.makedirs(AUDIO_DIR, exist_ok=True)

# Three test samples of increasing length
samples = {
    "short": (
        "The quick brown fox jumps over the lazy dog. "
        "This is a test of automatic speech recognition."
    ),
    "medium": (
        "Artificial intelligence has transformed the way we interact with technology. "
        "Machine learning models can now understand and generate human language with remarkable accuracy. "
        "Speech recognition systems like Whisper have made it possible to transcribe audio in real time, "
        "supporting dozens of languages and handling noisy environments with ease. "
        "The combination of hardware acceleration and optimized software makes these capabilities "
        "accessible at scale, enabling applications from live captioning to voice assistants."
    ),
    "long": (
        "The history of artificial intelligence is a fascinating journey through decades of research and discovery. "
        "In the nineteen fifties, pioneers like Alan Turing asked whether machines could think, "
        "laying the philosophical groundwork for an entirely new field of computer science. "
        "Early systems used symbolic reasoning and hand-crafted rules to solve narrowly defined problems. "
        "The nineteen eighties saw the rise of expert systems, which encoded human knowledge into decision trees. "
        "But it was the advent of deep learning in the twenty tens that truly revolutionized the field. "
        "Neural networks with millions of parameters could learn directly from data, "
        "outperforming traditional approaches on tasks from image classification to natural language processing. "
        "Today, large language models and multimodal systems represent the cutting edge, "
        "capable of understanding text, images, and speech simultaneously. "
        "Custom silicon like AWS Trainium and Inferentia accelerates these workloads, "
        "making advanced AI accessible and cost-effective for organizations of every size."
    ),
}

audio_files = {}
for name, text in samples.items():
    wav_path = os.path.join(AUDIO_DIR, f"{name}.wav")
    if not os.path.exists(wav_path):
        mp3_path = os.path.join(AUDIO_DIR, f"{name}.mp3")
        tts = gTTS(text, lang="en")
        tts.save(mp3_path)
        subprocess.run(
            [FFMPEG, "-y", "-i", mp3_path, "-ar", "16000", "-ac", "1", wav_path],
            capture_output=True, check=True,
        )
    data, sr = sf.read(wav_path)
    duration = len(data) / sr
    audio_files[name] = {"path": wav_path, "duration": duration, "reference": text}
    print(f"{name:7s}: {duration:.1f}s — {wav_path}")

short  : 7.3s — /tmp/whisper_audio/short.wav


medium : 38.9s — /tmp/whisper_audio/medium.wav


long   : 83.0s — /tmp/whisper_audio/long.wav


## 9. Transcribe

Run transcription on all test samples. The NxDI model exposes the same `transcribe()` API as OpenAI Whisper.

In [10]:
# Warmup run
_ = model.transcribe(audio_files["short"]["path"], language="en", verbose=False)

for name, info in audio_files.items():
    result = model.transcribe(info["path"], language="en", verbose=False)
    info["neuron_text"] = result["text"].strip()
    print(f"\n--- {name} ({info['duration']:.1f}s) ---")
    print(f"Neuron: {info['neuron_text']}")

/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


  0%|          | 0/732 [00:00<?, ?frames/s]

100%|██████████| 732/732 [00:00<00:00, 1749.47frames/s]

100%|██████████| 732/732 [00:00<00:00, 1745.19frames/s]


/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


...

 33%|███▎      | 2700/8296 [00:00<00:00, 10631.12frames/s]

 68%|██████▊   | 5640/8296 [00:00<00:00, 10610.27frames/s]

 99%|█████████▉| 8216/8296 [00:00<00:00, 10461.19frames/s]

100%|██████████| 8296/8296 [00:03<00:00, 2105.74frames/s] 


--- long (83.0s) ---
Neuron: The history of artificial intelligence is a fascinating journey through decades of research and discovery. In the 1950s, pioneers like Alan Turing asked whether machines could think, laying the philosophical groundwork for an entirely new field of computer science. Early systems used symbolic reasoning and hand-crafted rules to solve narrowly defined problemsThe 1980s saw the rise of expert systems, which encoded human knowledge into decision trees. But it was the advent of deep learning in the 2010s that truly revolutionized the field. Neural networks with millions of parameters could learn directly from data, outperforming traditional approaches on tasks from image classification to natural language. Processing. Today, large language models and multi-modal systems represent the cutting edge. Capable of understanding text. Images. And speech simultaneously. Custom silicon like AWS Trainium and Inferentia accelerates these workloads. Making advanced AI acc

## 10. Benchmark

Measure latency across 10 runs per audio sample.

In [11]:
NUM_RUNS = 10

for name, info in audio_files.items():
    latencies = []
    for _ in range(NUM_RUNS):
        t0 = time.time()
        _ = model.transcribe(info["path"], language="en", verbose=False)
        latencies.append(time.time() - t0)
    info["neuron_avg_latency"] = np.mean(latencies)
    info["neuron_min_latency"] = min(latencies)
    info["neuron_rtf"] = info["duration"] / info["neuron_avg_latency"]

print(f"{'Audio':<8} {'Duration':>8} {'Avg Latency':>12} {'Min Latency':>12} {'RTF':>6}")
print("-" * 50)
for name, info in audio_files.items():
    print(f"{name:<8} {info['duration']:>7.1f}s {info['neuron_avg_latency']:>11.3f}s {info['neuron_min_latency']:>11.3f}s {info['neuron_rtf']:>5.1f}x")

/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


  0%|          | 0/732 [00:00<?, ?frames/s]

100%|██████████| 732/732 [00:00<00:00, 5554.76frames/s]

100%|██████████| 732/732 [00:00<00:00, 5522.25frames/s]


/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


...

 33%|███▎      | 2700/8296 [00:00<00:00, 11099.76frames/s]

 68%|██████▊   | 5640/8296 [00:00<00:00, 11228.40frames/s]

 99%|█████████▉| 8216/8296 [00:00<00:00, 11078.49frames/s]

100%|██████████| 8296/8296 [00:03<00:00, 2092.47frames/s] 

Audio    Duration  Avg Latency  Min Latency    RTF
--------------------------------------------------
short        7.3s       0.176s       0.173s  41.6x
medium      38.9s       0.459s       0.450s  84.7x
long        83.0s       4.272s       3.624s  19.4x


## 11. CPU Comparison & Accuracy

Run the same audio through OpenAI Whisper on CPU. Compare transcription accuracy using Word Error Rate (WER).

In [12]:
import whisper
from jiwer import wer

cpu_model = whisper.load_model("large-v3-turbo", device="cpu")

for name, info in audio_files.items():
    t0 = time.time()
    cpu_result = cpu_model.transcribe(info["path"], language="en", verbose=False)
    info["cpu_latency"] = time.time() - t0
    info["cpu_text"] = cpu_result["text"].strip()
    info["speedup"] = info["cpu_latency"] / info["neuron_avg_latency"]
    info["wer"] = wer(info["cpu_text"], info["neuron_text"])

print(f"{'Audio':<8} {'Neuron':>10} {'CPU':>10} {'Speedup':>8} {'WER':>6}")
print("-" * 46)
for name, info in audio_files.items():
    print(f"{name:<8} {info['neuron_avg_latency']:>9.3f}s {info['cpu_latency']:>9.3f}s {info['speedup']:>7.1f}x {info['wer']:>5.1%}")

print("\n--- Transcription Comparison ---")
for name, info in audio_files.items():
    match = "MATCH" if info["neuron_text"] == info["cpu_text"] else "DIFF"
    print(f"\n[{name}] ({match})")
    print(f"  Neuron: {info['neuron_text']}")
    print(f"  CPU:    {info['cpu_text']}")

  0%|                                              | 0.00/1.51G [00:00<?, ?iB/s]

  0%|                                     | 1.28M/1.51G [00:00<02:07, 12.7MiB/s]

  1%|▎                                    | 14.3M/1.51G [00:00<00:19, 83.8MiB/s]

  2%|▉                                     | 38.0M/1.51G [00:00<00:10, 158MiB/s]

  3%|█▎                                    | 53.5M/1.51G [00:00<00:09, 160MiB/s]

...

 99%|█████████▉| 8216/8296 [00:09<00:00, 832.27frames/s]

 99%|█████████▉| 8216/8296 [00:27<00:00, 832.27frames/s]

100%|██████████| 8296/8296 [00:35<00:00, 138.44frames/s]

100%|██████████| 8296/8296 [00:35<00:00, 233.91frames/s]

Audio        Neuron        CPU  Speedup    WER
----------------------------------------------
short        0.176s     5.654s    32.1x  0.0%
medium       0.459s     5.948s    12.9x  0.0%
long         4.272s    35.538s     8.3x 32.7%

--- Transcription Comparison ---

[short] (MATCH)
  Neuron: The quick brown fox jumps over the lazy dog. This is a test of automatic speech recognition.
  CPU:    The quick brown fox jumps over the lazy dog. This is a test of automatic speech recognition.

[medium] (MATCH)
  Neuron: Artificial intelligence has transformed the way we interact with technology. Machine learning models can now understand and generate human language with remarkable accuracy. Speech recognition systems like Whisper have made it possible to transcribe audio in real time. Supporting dozens of languages and handling noisy environments with ease. The combination of hardware acceleration and optimized software makes these capabilities accessible. At scale. Enabling applications from l

## 12. Results Summary

In [13]:
# Collect all results into a summary dict
results_summary = {
    "platform": PLATFORM,
    "model": "openai/whisper-large-v3-turbo",
    "precision": "FP16",
    "tp_degree": 1,
    "compile_time_s": compile_time,
    "load_time_s": round(load_time, 1),
    "benchmarks": {},
}

for name, info in audio_files.items():
    results_summary["benchmarks"][name] = {
        "audio_duration_s": round(info["duration"], 1),
        "neuron_avg_latency_s": round(info["neuron_avg_latency"], 3),
        "neuron_min_latency_s": round(info["neuron_min_latency"], 3),
        "rtf": round(info["neuron_rtf"], 1),
        "cpu_latency_s": round(info["cpu_latency"], 3),
        "speedup_vs_cpu": round(info["speedup"], 1),
        "wer_vs_cpu": round(info["wer"], 4),
        "neuron_text": info["neuron_text"],
        "cpu_text": info["cpu_text"],
    }

# Save to JSON
results_file = f"/tmp/whisper_results_{PLATFORM}.json"
with open(results_file, "w") as f:
    json.dump(results_summary, f, indent=2)
print(f"Results saved to {results_file}")
print(json.dumps(results_summary, indent=2))

Results saved to /tmp/whisper_results_trn2.json
{
  "platform": "trn2",
  "model": "openai/whisper-large-v3-turbo",
  "precision": "FP16",
  "tp_degree": 1,
  "compile_time_s": 27.929370403289795,
  "load_time_s": 7.7,
  "benchmarks": {
    "short": {
      "audio_duration_s": 7.3,
      "neuron_avg_latency_s": 0.176,
      "neuron_min_latency_s": 0.173,
      "rtf": 41.6,
      "cpu_latency_s": 5.654,
      "speedup_vs_cpu": 32.1,
      "wer_vs_cpu": 0.0,
      "neuron_text": "The quick brown fox jumps over the lazy dog. This is a test of automatic speech recognition.",
      "cpu_text": "The quick brown fox jumps over the lazy dog. This is a test of automatic speech recognition."
    },
    "medium": {
      "audio_duration_s": 38.9,
      "neuron_avg_latency_s": 0.459,
      "neuron_min_latency_s": 0.45,
      "rtf": 84.7,
      "cpu_latency_s": 5.948,
      "speedup_vs_cpu": 12.9,
      "wer_vs_cpu": 0.0,
      "neuron_text": "Artificial intelligence has transformed the way we inte

## 13. Transcribe Your Own Audio

Replace the path below with any audio file (WAV, MP3, FLAC, etc.).

In [14]:
# my_audio = "/path/to/your/audio.wav"
# result = model.transcribe(my_audio, language="en", verbose=True)
# print(result["text"])

## Performance Results

*(Results will be filled in after testing on both platforms)*

### trn2.3xlarge (LNC=2, SDK 2.28)

| Audio | Duration | Neuron Latency | RTF | CPU Latency | Speedup | WER |
|-------|----------|---------------|-----|-------------|---------|-----|
| short | | | | | | |
| medium | | | | | | |
| long | | | | | | |

- Compilation: s
- Model load: s

### inf2.xlarge (SDK 2.28)

| Audio | Duration | Neuron Latency | RTF | CPU Latency | Speedup | WER |
|-------|----------|---------------|-----|-------------|---------|-----|
| short | | | | | | |
| medium | | | | | | |
| long | | | | | | |

- Compilation: s
- Model load: s

### Optimizations Applied

| Optimization | Description | Impact |
|---|---|---|
| Cross-attn K/V cache | Skip redundant K/V projections during decode | ~2.5x decode latency reduction |
| Fused QKV | Single matmul for Q/K/V in self-attention | Reduced kernel launch overhead |
| NKI flash attention | Hardware flash attention in 32 encoder layers | 45% faster compilation |
| NKI fused Conv1D+GELU | Fused encoder frontend convolutions | Marginal (encoder frontend not bottleneck) |

All NKI kernels are from existing libraries (NxDI SDK and nki-library) — no custom kernels.